In [ ]:
import jax
import jax.numpy as jnp
from transformers import AutoTokenizer, AutoProcessor
from tunix.generate import sampler
from tunix.models.gemma4 import model
from tunix.models.gemma4 import params_safetensors


MESH = [(1, 2), ("fsdp", "tp")]
mesh = jax.make_mesh(
    *MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0])
)

# Download HF safetensors to your local dir.
# e.g. hf download google/gemma-4-31B-it --local-dir <your local path>
E2B_PATH = "<your local path>"
E4B_PATH = "<your local path>"
MOE_26B_PATH = "<your local path>"
DENSE_31B_PATH = "<your local path>"

version = "e2b"
if version == "e2b":
  tokenizer = AutoTokenizer.from_pretrained(E2B_PATH)
  config = model.ModelConfig.gemma4_e2b()
  model = params_safetensors.create_model_from_safe_tensors(
      E2B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=35, num_kv_heads=1, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(E2B_PATH)
elif version == "e4b":
  tokenizer = AutoTokenizer.from_pretrained(E4B_PATH)
  config = model.ModelConfig.gemma4_e4b()
  model = params_safetensors.create_model_from_safe_tensors(
      E4B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=42, num_kv_heads=2, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(E4B_PATH)
elif version == "moe_26b":
  tokenizer = AutoTokenizer.from_pretrained(MOE_26B_PATH)
  config = model.ModelConfig.gemma4_26b_a4b()
  model = params_safetensors.create_model_from_safe_tensors(
      MOE_26B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=30, num_kv_heads=8, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(MOE_26B_PATH)
elif version == "31b":
  tokenizer = AutoTokenizer.from_pretrained(DENSE_31B_PATH)
  config = model.ModelConfig.gemma4_31b()
  model = params_safetensors.create_model_from_safe_tensors(
      DENSE_31B_PATH, config, mesh, dtype=jnp.bfloat16
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=60, num_kv_heads=16, head_dim=512
  )
else:
  raise ValueError(f"Unknown version: {version}")

In [ ]:
def templatize(prompts):
  out = []
  for p in prompts:
    out.append(
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": p},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )
  return out


inputs = templatize([
    "which is larger 9.9 or 9.11?",
    "why sky is blue?",
    "write code to solve fibonacci in python",
    "Write a short joke about saving RAM.",
])

sampler = sampler.Sampler(
    model,
    tokenizer,
    cache_config=cache_config,
)
with mesh:
  out = sampler(
      inputs,
      max_generation_steps=1024,
      echo=True,
      temperature=0.6,
      eos_tokens=[1, 106, 50],
  )

  for t in out.text:
    print(t)
    print("*" * 30)

## Vision example

import tensorflow_datasets as tfds

ds = tfds.data_source('oxford_flowers102', split='train')
image1 = ds[0]['image']
image2 = ds[1]['image']
image3 = ds[2]['image']

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image1)

In [ ]:
plt.imshow(image2)

In [ ]:
messages = [[
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the images."},
            {"type": "image"},
            {"type": "image"},
        ]
    }],
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Write a poem for the image."},
            {"type": "image"},
        ]
    }]
]

prompt = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

with mesh:
    output = sampler(
        input_strings=prompt,
        images=[[image1, image2], [image3]],
        max_generation_steps=128,
        temperature=1,
        echo=False,
        eos_tokens=[1, 106, 50],
    )
for o in output.text:
    print(o)
    print("####" * 8)

## Audio Example

In [ ]:
%pip install audioop-lts

In [ ]:
import tensorflow_datasets as tfds
import numpy as np
import IPython.display as ipd

ds, info = tfds.load('crema_d', split='train', with_info=True)
assert info.features['audio'].sample_rate == 16000  # Gemma4 needs 16kHz

audios = [x['audio'].numpy().astype(np.float32) for x in list(ds.take(4))]
audios = [x / np.max(np.abs(x)) for x in audios]  # normalize

for audio in audios:
  ipd.display(ipd.Audio(audio, rate=16000))

In [ ]:
messages = [[
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcribe the following speech segment in English into English text."},
            {"type": "audio"},
        ]
    }],
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the tone of the speaker."},
            {"type": "audio"},
        ]
    }],
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcribe the following speech segment in English into English text."},
            {"type": "audio"},
            {"type": "audio"},
        ]
    }],
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Why is the sky blue?"},
        ]
    }]
]

prompt = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

with mesh:
    output = sampler(
        input_strings=prompt,
        audios=[[audios[0]], [audios[1]], [audios[0], audios[2]], []],
        max_audio_clips=2,
        max_generation_steps=128,
        temperature=1,
        echo=False,
        eos_tokens=[1, 106, 50],
    )
for o in output.text:
    print(o)
    print("####" * 8)